# 04 — Training and Evaluation

Trains one model per stellar parameter (Teff, logg, [Fe/H]) using the
best pipeline from tuning, evaluates on a held-out test set, and saves
trained models and prediction results.

**Hyperparameter source — choose one:**
- `HYPERPARAMS_SOURCE = 'tuning'` — load the best pipeline from `03_tuning.ipynb`
- `HYPERPARAMS_SOURCE = 'manual'` — define hyperparameters directly in this notebook

**Output:**
- `models/<MODEL_TYPE>/<training_id>_<param>.<json|sav>` — trained model
- `predicts/<MODEL_TYPE>/<training_id>_<param>.csv` — test set predictions
- `predicts/<MODEL_TYPE>/<training_id>_<param>_metrics.json` — R² and MAD

## Configuration

In [ ]:
import os
import json
import joblib
import pandas as pd
import numpy as np
import minas as mg

# ── Files ─────────────────────────────────────────────────────────────────────
INPUT_FILE   = 'input_catalog.csv'  # output from 01_data_creation.ipynb
INDEX_COL    = 'ID'
PIPELINE_DIR = 'pipelines'          # folder with .joblib files from 03_tuning.ipynb
MODELS_DIR   = 'models'
PREDICTS_DIR = 'predicts'

# ── Survey ────────────────────────────────────────────────────────────────────
SURVEY = 'SPLUS'

# ── Parameters to train ──────────────────────────────────────────────────────
PARAM_LIST = ['teff', 'logg', 'feh']

# ── Distance column ───────────────────────────────────────────────────────────
DIST_COL = 'Dist'   # set to None if absolute magnitudes are not needed

# ── Model type ────────────────────────────────────────────────────────────────
MODEL_TYPE = 'XGB'   # 'RF' or 'XGB'

# ── Hyperparameter source ────────────────────────────────────────────────────
# 'tuning' → load best pipeline from 03_tuning.ipynb (recommended)
# 'manual' → define hyperparameters below
HYPERPARAMS_SOURCE = 'tuning'
TUNING_ID          = ''   # datetime string from 03_tuning.ipynb (e.g. '20251027164829')
                           # leave empty to use the most recent pipeline found

# ── Feature selection ─────────────────────────────────────────────────────────
# False → use all features
# True  → load selected features from 02_feature_importance.ipynb
USE_FEATURE_SELECTION = False
FEATURES_DIR          = 'features'

# ── Train/test split ──────────────────────────────────────────────────────────
TEST_SIZE    = 0.25
RANDOM_STATE = 42

# ── Evaluation bins (for MAD per bin) ────────────────────────────────────────
BINS = {
    'teff': [3000, 4000, 5000, 6000, 7000, 8000],
    'logg': [1.0, 2.0, 3.0, 4.0, 5.0],
    'feh' : [-2.5, -1.5, -0.5, 0.5, 1.0],
}

datetime_str = pd.Timestamp.now().strftime('%Y%m%d%H%M%S')
os.makedirs(MODELS_DIR,   exist_ok=True)
os.makedirs(PREDICTS_DIR, exist_ok=True)
print(f'Training ID : {datetime_str}')
print(f'Model type  : {MODEL_TYPE}')
print(f'HP source   : {HYPERPARAMS_SOURCE}')

## Manual hyperparameters (only used when `HYPERPARAMS_SOURCE = 'manual'`)

In [ ]:
# Edit these values if HYPERPARAMS_SOURCE = 'manual'
MANUAL_HYPERPARAMS = {
    'teff': {
        'colsample_bytree': 0.8,
        'gamma'           : 0.1,
        'learning_rate'   : 0.05,
        'max_depth'       : 6,
        'n_estimators'    : 300,
        'subsample'       : 0.8,
        'random_state'    : RANDOM_STATE,
    },
    'logg': {
        'colsample_bytree': 0.8,
        'gamma'           : 0.1,
        'learning_rate'   : 0.05,
        'max_depth'       : 6,
        'n_estimators'    : 300,
        'subsample'       : 0.8,
        'random_state'    : RANDOM_STATE,
    },
    'feh': {
        'colsample_bytree': 0.8,
        'gamma'           : 0.1,
        'learning_rate'   : 0.05,
        'max_depth'       : 6,
        'n_estimators'    : 300,
        'subsample'       : 0.8,
        'random_state'    : RANDOM_STATE,
    },
}

## Step 1 — Load catalog and assemble feature DataFrames

In [ ]:
def get_column(df, aliases):
    """Return the first matching column from a list of aliases."""
    for col in aliases:
        if col in df.columns:
            return col
    raise KeyError(f'No column found for aliases: {aliases}')


df_base = pd.read_csv(INPUT_FILE)
if INDEX_COL and INDEX_COL in df_base.columns:
    df_base = df_base.set_index(INDEX_COL)

filters = mg.FILTERS[SURVEY]
print(f'Catalog loaded: {len(df_base):,} objects')

preprocessed = {}
for param in PARAM_LIST:
    df = df_base.copy()

    # Remove invalid values
    param_col = get_column(df, mg.PARAM_ALIASES[param])
    df = df[df[param_col] != -9999].dropna(subset=[param_col])

    # Compute absolute magnitudes
    if DIST_COL and DIST_COL in df.columns:
        df = mg.preprocess.calculate_abs_mag(df, filters, DIST_COL)

    # Assemble feature DataFrame
    work_df = mg.preprocess.assemble_work_df(
        df=df,
        filters=filters,
        correction_pairs=None,
        add_colors=True,
        verbose=False,
    )

    # Apply feature selection if requested
    if USE_FEATURE_SELECTION:
        feat_path = os.path.join(FEATURES_DIR, f'{param}_features.json')
        if not os.path.exists(feat_path):
            raise FileNotFoundError(f'Feature file not found: {feat_path}\nRun 02_feature_importance.ipynb first.')
        with open(feat_path) as f:
            selected = json.load(f)
        work_df = work_df[selected]

    preprocessed[param] = {'df': df, 'work_df': work_df, 'param_col': param_col}
    feat_label = f'{work_df.shape[1]} features (selected)' if USE_FEATURE_SELECTION else f'{work_df.shape[1]} features (all)'
    print(f'  [{param:4s}] {len(df):,} objects  |  {feat_label}')

## Step 2 — Load or create pipelines

In [ ]:
pipelines = {}

for param in PARAM_LIST:
    if HYPERPARAMS_SOURCE == 'tuning':
        # Find the pipeline file in PIPELINE_DIR
        all_files = [f for f in os.listdir(PIPELINE_DIR)
                     if f.endswith('.joblib') and f'_{param}_' in f and MODEL_TYPE in f]

        if TUNING_ID:
            all_files = [f for f in all_files if TUNING_ID in f]

        if not all_files:
            raise FileNotFoundError(
                f'No pipeline found for [{param}] in {PIPELINE_DIR}.\n'
                f'Run 03_tuning.ipynb first or set HYPERPARAMS_SOURCE = "manual".'
            )

        # Use the most recent file
        pipeline_file = sorted(all_files)[-1]
        pipeline = joblib.load(os.path.join(PIPELINE_DIR, pipeline_file))
        print(f'  [{param:4s}] loaded pipeline: {pipeline_file}')

    else:  # manual
        hp       = MANUAL_HYPERPARAMS[param]
        model_key = 'RF-REG' if MODEL_TYPE == 'RF' else 'XGB-REG'
        pipeline = mg.models.create_model(model_key, hp_combination=tuple(hp.values()))
        print(f'  [{param:4s}] created pipeline with manual hyperparameters')

    pipelines[param] = pipeline

## Step 3 — Train and evaluate

In [ ]:
print('=== TRAINING ===')
training_results = {}

for i, param in enumerate(PARAM_LIST):
    print(f'\n[{i+1}/{len(PARAM_LIST)}] {param}')

    df        = preprocessed[param]['df']
    work_df   = preprocessed[param]['work_df']
    param_col = preprocessed[param]['param_col']
    pipeline  = pipelines[param]

    # Align indices
    idx     = df.index.intersection(work_df.index)
    df_sync = df.loc[idx]
    X       = work_df.loc[idx]
    Y       = df_sync[param_col]

    # Train / test split
    X_train, X_test, Y_train, Y_test = mg.train_test_split(
        X, Y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    pipeline.fit(X_train, Y_train)
    Y_pred = pipeline.predict(X_test)

    # Metrics
    mad = mg.metrics.mad(Y_test, Y_pred)
    r2  = mg.metrics.r2_score(Y_test, Y_pred)
    print(f'  R² = {r2:.4f}  |  MAD = {mad:.4f}')

    # Save predictions
    results_df = pd.DataFrame({'true': Y_test, 'predicted': Y_pred})
    if 'RA' in df_sync.columns and 'DEC' in df_sync.columns:
        results_df = results_df.join(df_sync.loc[Y_test.index, ['RA', 'DEC']])

    results_path = os.path.join(PREDICTS_DIR, f'{datetime_str}_{SURVEY}_{param}_{MODEL_TYPE}.csv')
    metrics_path = results_path.replace('.csv', '_metrics.json')

    results_df.to_csv(results_path)
    with open(metrics_path, 'w') as f:
        json.dump({'r2': float(r2), 'mad': float(mad)}, f, indent=2)

    print(f'  Predictions saved : {results_path}')
    print(f'  Metrics saved     : {metrics_path}')

    training_results[param] = {
        'results_path': results_path,
        'metrics_path': metrics_path,
        'param_col'   : param_col,
        'r2': r2, 'mad': mad,
    }

print(f'\nTraining ID: {datetime_str}')

## Step 4 — Save trained models

In [ ]:
print('=== SAVING MODELS ===')

for param in PARAM_LIST:
    pipeline   = pipelines[param]
    model_path = os.path.join(MODELS_DIR, f'{datetime_str}_{SURVEY}_{param}_{MODEL_TYPE}')

    # XGBoost: extract booster and save as .json
    xgb_model = None
    if hasattr(pipeline, 'named_steps') and 'xgbregressor' in pipeline.named_steps:
        xgb_model = pipeline.named_steps['xgbregressor']
    elif hasattr(pipeline, 'save_model'):
        xgb_model = pipeline

    if xgb_model is not None:
        xgb_model.save_model(model_path + '.json')
        print(f'  [{param:4s}] saved: {model_path}.json')
    else:
        joblib.dump(pipeline, model_path + '.sav')
        print(f'  [{param:4s}] saved: {model_path}.sav')

print('\nAll models saved.')

## Step 5 — Evaluation plots

In [ ]:
from minas.evaluation._graphics import plot_test_graphs

PARAM_UNITS = {'teff': 'K', 'logg': 'dex', 'feh': 'dex'}
CMAPS       = {'teff': 'Reds', 'logg': 'Greens', 'feh': 'Blues'}

graphs_dir = 'graphs'
os.makedirs(graphs_dir, exist_ok=True)

# Print metrics summary
print(f'{'Parameter':>10}  {'R²':>8}  {'MAD':>8}')
print('-' * 32)
for param in PARAM_LIST:
    r = training_results[param]
    print(f'{param:>10}  {r["r2"]:>8.4f}  {r["mad"]:>8.4f}')

# Generate scatter + KDE plots
for param in PARAM_LIST:
    r          = training_results[param]
    results_df = pd.read_csv(r['results_path'], index_col=0)

    fig = plot_test_graphs(
        predictions=results_df['predicted'],
        true_values=results_df['true'],
        bins=BINS[param],
        cmap_name=CMAPS[param],
        param_string=param,
        param_unit=PARAM_UNITS[param],
        n_ticks=5,
        legend=True,
    )

    graph_path = os.path.join(graphs_dir, f'{datetime_str}_{SURVEY}_{param}_{MODEL_TYPE}.png')
    fig.savefig(graph_path, bbox_inches='tight', dpi=150)
    print(f'Graph saved: {graph_path}')
    mg.evaluation.show(fig)